# Study 815 — Variance-Ratio Reversal — the teardown

The Lo-MacKinlay overlapping VR(5) signal, the per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation placebo, the two-era and two-window robustness cuts, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_days': 4026, 'median_names': 50, 'fingerprint': '357fd262912f', 'spread_bps': -2.69, 't_nw': -2.44, 't_1s': -2.44, 'lo_bps': 6.1, 'hi_bps': 8.8, 'welch_t': -1.05, 'gross_sharpe': -0.61, 'vr_median': 0.991, 'vr_pct_below': 52, 'vr_min': 0.615, 'vr_max': 1.526, 'placebo_obs': -2.69, 'placebo_mean': 0.095, 'placebo_sd': 0.937, 'placebo_p': 0.999, 'placebo_sd_from_centre': -2.88, 'placebo_draws': 1000, 'era_early_bps': -3.04, 'era_early_t': -2.42, 'era_early_n': 1892, 'era_late_bps': -2.39, 'era_late_t': -1.36, 'era_late_n': 2134, 'win63_bps': -0.68, 'win63_t': -0.66, 'win252_bps': -0.57, 'win252_t': -0.53, 'timer_1_gross': -2.69, 'timer_1_cost': 2.14, 'timer_1_net': -4.83, 'timer_1_t': -4.38, 'timer_1_sharpe': -1.1, 'timer_1_ann': -12.2, 'timer_5_gross': -2.69, 'timer_5_cost': 10.14, 'timer_5_net': -12.83, 'timer_5_t': -11.64, 'timer_5_sharpe': -2.91, 'timer_5_ann': -32.3, 'null_mean_t': 0.05, 'null_sd_t': 0.83, 'null_fire': 1, 'planted_t': 9.75, 'planted_welch': 8.76}

## The headline — long-low-VR / short-high-VR spread

Daily equal-weight bottom-30% (low VR, mean-reverting) minus top-30% (high VR, trending) forward-return spread. VR = Lo-MacKinlay overlapping, bias-corrected, trailing 120 days, q=5.

In [2]:
print(f"cross-section : VR(5) median {R['vr_median']:.3f}, range [{R['vr_min']:.3f}, {R['vr_max']:.3f}], {R['vr_pct_below']}% below 1")
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : low-VR {R['lo_bps']:+.2f} vs high-VR {R['hi_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

cross-section : VR(5) median 0.991, range [0.615, 1.526], 52% below 1
spread        : -2.69 bps/day  NW(10) t = -2.44  one-sample t = -2.44
books         : low-VR +6.10 vs high-VR +8.80 bps (Welch t = -1.05)
gross Sharpe  : -0.61 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations)

Keep the VR sort, break the signal→forward-return link. The observed spread sits deep in the *left* tail — the (opposite-sign) relation is not a lucky sort, it is simply the reverse of what the reversal story predicts.

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> right-tail p = {R['placebo_p']:.5f}")
print(f"observed sits {R['placebo_sd_from_centre']:+.2f} sd from the null centre (left tail)")

observed -2.69 bps vs placebo mean +0.095 (sd 0.937) -> right-tail p = 0.99900
observed sits -2.88 sd from the null centre (left tail)


## Robustness — two eras and two windows

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print(f"window  63d           : {R['win63_bps']:+.2f} bps  NW t = {R['win63_t']:+.2f}")
print(f"window 252d           : {R['win252_bps']:+.2f} bps  NW t = {R['win252_t']:+.2f}")

2010-2017 (n=1892): -3.04 bps  NW t = -2.42
2018-2026 (n=2134): -2.39 bps  NW t = -1.36
window  63d           : -0.68 bps  NW t = -0.66
window 252d           : -0.57 bps  NW t = -0.53


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -2.69 -> net -4.83 bps/day (cost 2.14/day, t=-4.38)
5 bps one-way: gross -2.69 -> net -12.83 bps/day (cost 10.14/day, t=-11.64)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted low-VR reversal premium.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from variance_ratio import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=815+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0006, seed=815, n_assets=40, n_days=1600))
print(f"planted (edge=0.0006): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (edge=0), 8 seeds: NW t mean +0.37 (sd 0.74), |t|>=2 in 0/8


planted (edge=0.0006): NW t = +9.75, Welch t = +8.76


## Verdict

- **Signal — None.** The claimed variance-ratio reversal premium does **not** replicate on 50 liquid US mega-caps: the long-low-VR / short-high-VR spread is **-2.69 bps/day** (NW *t* = **-2.44**) — significant at the headline 120-day window but *opposite in sign* (the trending high-VR names out-earned the mean-reverters), and **fragile**: the 2018–2026 era is insignificant (*t* = -1.36) and both the 63-day and 252-day VR windows vanish (*t* = -0.66 / -0.53). The 20-seed synthetic control recovers a *planted* low-VR premium cleanly (*t* = +9.75, fires on 1/20 nulls), so the null result is not a broken engine — there is simply no mean-reversion premium here. Survivorship biases the magnitude.
- **Tradability — Mirage.** Even the sign-flipped book dies: at 1 bp one-way the friction (2.14 bps/day) already dwarfs the 2.69 bps gross edge, net **-4.83 bps/day** (*t* = -4.38); at 5 bps **-12.83 bps/day**.